# QAI Control — Preparação de Dados e Formulação do Problema Preditivo

## Objetivo

Este notebook apresenta a preparação dos dados do projeto **QAI Control** e incorpora os ajustes solicitados no feedback do Deliverable 1.



**Relatório de mudanças realizadas**

Com base no feedback do professor, foram feitas alterações na preparação dos dados e na formulação do problema preditivo do projeto.

A principal melhoria foi a definição clara do problema de Machine Learning. O projeto passou a considerar como objetivo a previsão do Risk Score de Dependência de IA do colaborador no mês seguinte, em uma escala de 0 a 100. A unidade de análise foi definida como colaborador por mês, a tarefa como regressão e a métrica principal como MAE (Mean Absolute Error). Essa escolha permite interpretar o erro diretamente em pontos do score, facilitando a utilização do resultado pelo gestor para priorização de casos de maior risco.

Também foi definido um corte temporal entre treino, validação e teste, evitando o uso de informações futuras durante o treinamento. Os preditores foram revisados para garantir que representem apenas informações disponíveis até o momento da previsão, reduzindo o risco de vazamento de dados.

A construção do Risk Score foi formalizada utilizando apenas variáveis existentes na base: frequência de uso, intensidade de requisições, volume de tokens e concentração do uso em ferramentas de IA. Os componentes foram transformados em percentis mensais e combinados com pesos iguais. Foi registrado que esse score é uma proxy operacional criada para o projeto, pois a base não possui um rótulo real de dependência de IA.

A conversão cambial também foi revisada. As taxas anteriormente utilizadas, que alternavam entre valores praticamente constantes e não possuíam fonte documentada, foram substituídas por uma série oficial do Banco Central do Brasil. Como a tabela de faturamento possui apenas o mês de referência, foi adotada a média das cotações diárias do dólar para cada mês. Os valores originais de custo e moeda continuam preservados para garantir rastreabilidade, auditoria e possibilidade de refazer a conversão.

Por fim, foram mantidas e reforçadas as decisões já avaliadas positivamente: tratamento de latency_ms como ausência informativa, identificação de NaNs estruturais e preservação dos dados originais antes das transformações.

Com essas alterações, a preparação deixa de ser apenas descritiva e passa a estar diretamente alinhada ao problema preditivo e à futura etapa de modelagem.

In [2]:
import zipfile
from pathlib import Path

import numpy as np
import pandas as pd


try:
    from google.colab import files

    print("Selecione o arquivo .zip da base:")
    uploaded = files.upload()

    zip_names = [name for name in uploaded.keys() if name.lower().endswith(".zip")]

    if not zip_names:
        raise FileNotFoundError("Nenhum arquivo .zip foi enviado.")

    ZIP_PATH = Path(zip_names[0])

except ImportError:
    # Execução fora do Colab
    candidatos = [
        Path("synthetic_ai_usage_dataset (1) (1).zip"),
        Path("/mnt/data/synthetic_ai_usage_dataset (1) (1).zip"),
    ]
    ZIP_PATH = next((p for p in candidatos if p.exists()), None)

    if ZIP_PATH is None:
        raise FileNotFoundError(
            "Arquivo .zip não encontrado. Coloque-o na mesma pasta do notebook."
        )

print(f"Arquivo utilizado: {ZIP_PATH}")

EXTRACT_DIR = Path("qai_dataset")
EXTRACT_DIR.mkdir(exist_ok=True)

with zipfile.ZipFile(ZIP_PATH, "r") as z:
    z.extractall(EXTRACT_DIR)

csv_files = list(EXTRACT_DIR.rglob("*.csv"))

if not csv_files:
    raise FileNotFoundError("Nenhum arquivo CSV foi encontrado dentro do .zip.")

CSV_DIR = csv_files[0].parent

dfs = {
    arquivo.name: pd.read_csv(arquivo, low_memory=False)
    for arquivo in sorted(CSV_DIR.glob("*.csv"))
}

print(f"\n{len(dfs)} tabelas carregadas com sucesso.")
print("Pasta dos CSVs:", CSV_DIR)
sorted(dfs.keys())


Selecione o arquivo .zip da base:


Saving synthetic_ai_usage_dataset.zip to synthetic_ai_usage_dataset (1).zip
Arquivo utilizado: synthetic_ai_usage_dataset (1).zip

17 tabelas carregadas com sucesso.
Pasta dos CSVs: qai_dataset/synthetic_ai_usage_dataset/csv


['ai_billing.csv',
 'ai_budgets.csv',
 'ai_licenses.csv',
 'ai_models.csv',
 'ai_tools.csv',
 'ai_usage_logs.csv',
 'approved_ai_tools.csv',
 'builds.csv',
 'companies.csv',
 'departments.csv',
 'deployments.csv',
 'employees.csv',
 'issues.csv',
 'projects.csv',
 'pull_requests.csv',
 'security_events.csv',
 'training_records.csv']

## 1. Diagnóstico inicial da base

In [3]:
resumo_tabelas = []

for nome, df in sorted(dfs.items()):
    resumo_tabelas.append({
        "tabela": nome,
        "linhas": df.shape[0],
        "colunas": df.shape[1],
        "duplicatas": int(df.duplicated().sum()),
        "celulas_nulas": int(df.isna().sum().sum())
    })

resumo_tabelas = pd.DataFrame(resumo_tabelas)
display(resumo_tabelas)


,tabela,linhas,colunas,duplicatas,celulas_nulas
0,ai_billing.csv,7025,10,0,5684
1,ai_budgets.csv,2254,5,0,0
2,ai_licenses.csv,5605,7,0,1574
3,ai_models.csv,25,6,0,10
4,ai_tools.csv,15,5,0,0
5,ai_usage_logs.csv,269515,15,0,219221
6,approved_ai_tools.csv,105,5,0,48
7,builds.csv,12414,7,0,0
8,companies.csv,18,5,0,0
9,departments.csv,161,3,0,0


In [4]:
diagnostico = []

for nome, df in sorted(dfs.items()):
    for coluna in df.columns:
        diagnostico.append({
            "tabela": nome,
            "coluna": coluna,
            "tipo": str(df[coluna].dtype),
            "nulos_qtd": int(df[coluna].isna().sum()),
            "nulos_pct": round(df[coluna].isna().mean() * 100, 2),
            "cardinalidade": int(df[coluna].nunique(dropna=True))
        })

diagnostico = pd.DataFrame(diagnostico)

display(
    diagnostico
    .sort_values(["nulos_pct", "cardinalidade"], ascending=[False, False])
    .head(30)
)


,tabela,coluna,tipo,nulos_qtd,nulos_pct,cardinalidade
46,ai_usage_logs.csv,project_id,object,206909,76.77,18
5,ai_billing.csv,model_id,object,5258,74.85,10
52,approved_ai_tools.csv,allowed_departments,object,48,45.71,14
120,training_records.csv,completion_date,object,678,42.45,371
122,training_records.csv,score,float64,678,42.45,46
87,issues.csv,closed_at,object,1403,40.61,1966
20,ai_licenses.csv,expiration_date,object,1574,28.08,460
99,pull_requests.csv,merged_at,object,1227,23.95,3897
26,ai_models.csv,input_price_usd_per_1k_tokens,float64,5,20.00,8
27,ai_models.csv,output_price_usd_per_1k_tokens,float64,5,20.00,7


### Decisão

O diagnóstico é feito antes de qualquer transformação. A intenção é separar problemas de **estrutura**, **tipo**, **conteúdo** e **semântica**.

Os identificadores permanecem na base para relacionamento e rastreabilidade, mas não serão tratados como variáveis categóricas do modelo.


## 2. Análise dos valores ausentes

In [5]:
usage = dfs["ai_usage_logs.csv"].copy()

latency_por_status = (
    usage.groupby("request_status")["latency_ms"]
    .apply(lambda s: round(s.isna().mean() * 100, 2))
    .rename("latency_ms_nulo_pct")
)

display(latency_por_status)


,latency_ms_nulo_pct
request_status,
error,40.56
rate_limited,39.01
success,0.00
timeout,39.48


### `latency_ms`: ausência como sinal

A ausência de `latency_ms` se concentra em situações de falha (`error`, `timeout` e `rate_limited`). Por isso, ela não será apagada por uma imputação automática.

A decisão é:

- preservar o valor ausente;
- criar um indicador `latency_missing`;
- nas agregações mensais, calcular a latência média somente com valores observados e manter separadamente a taxa de ausência.

Assim, o modelo poderá receber a informação de que a latência não foi registrada sem inventar um valor de latência.


In [6]:
print("project_id ausente em ai_usage_logs:")
print(round(dfs["ai_usage_logs.csv"]["project_id"].isna().mean() * 100, 2), "%")

print("\nmodel_id ausente em ai_billing:")
print(round(dfs["ai_billing.csv"]["model_id"].isna().mean() * 100, 2), "%")

print("\nexpiration_date ausente em ai_licenses:")
print(round(dfs["ai_licenses.csv"]["expiration_date"].isna().mean() * 100, 2), "%")

print("\ncompletion_date ausente em training_records:")
print(round(dfs["training_records.csv"]["completion_date"].isna().mean() * 100, 2), "%")


project_id ausente em ai_usage_logs:
76.77 %

model_id ausente em ai_billing:
74.85 %

expiration_date ausente em ai_licenses:
28.08 %

completion_date ausente em training_records:
42.45 %


### NaN estrutural

Os campos acima não devem receber automaticamente média, mediana ou moda.

- `project_id`: nem todo uso de IA está ligado a projeto;
- `model_id` em faturamento: cobrança por licença/assento pode não ter um modelo associado;
- `expiration_date`: pode não existir para determinados acordos;
- `completion_date` e `score`: não existem quando o treinamento não foi concluído.

Nesses casos, o vazio pode significar **“não se aplica”**, e não necessariamente “dado perdido”.


## 3. Correção de tipos

In [7]:
clean = {nome: df.copy() for nome, df in dfs.items()}

colunas_data = {
    "ai_usage_logs.csv": ["timestamp"],
    "employees.csv": ["hire_date"],
    "ai_licenses.csv": ["assigned_date", "expiration_date"],
    "training_records.csv": ["enrollment_date", "completion_date"],
    "security_events.csv": ["timestamp"],
    "projects.csv": ["start_date"],
    "pull_requests.csv": ["created_at", "merged_at"],
    "issues.csv": ["created_at", "closed_at"],
    "builds.csv": ["timestamp"],
    "deployments.csv": ["timestamp"],
    "approved_ai_tools.csv": ["approval_date"],
}

for tabela, colunas in colunas_data.items():
    for coluna in colunas:
        clean[tabela][coluna] = pd.to_datetime(
            clean[tabela][coluna],
            errors="coerce"
        )

# billing_month representa mês, e não uma data diária.
clean["ai_billing.csv"]["billing_month"] = (
    clean["ai_billing.csv"]["billing_month"]
    .astype(str)
    .str[:7]
)

print(clean["ai_usage_logs.csv"]["timestamp"].dtype)
print(clean["employees.csv"]["hire_date"].dtype)
print(clean["ai_billing.csv"]["billing_month"].head())


datetime64[ns]
datetime64[ns]
0    2025-06
1    2025-06
2    2025-06
3    2025-06
4    2025-06
Name: billing_month, dtype: object


As colunas temporais são convertidas antes da análise de conteúdo. `billing_month` permanece no formato `AAAA-MM`, porque a própria base informa apenas o mês da cobrança.


## 4. Conversão USD → BRL com série oficial do Banco Central

O feedback apontou corretamente que as taxas usadas anteriormente alternavam entre dois valores praticamente constantes e não possuíam fonte documentada.

Nesta versão, elas são substituídas por dados oficiais do **Banco Central do Brasil (BCB)**.

Fonte: Banco Central do Brasil (BCB) — Sistema PTAX, utilizando as cotações oficiais do dólar americano para venda disponibilizadas pela API PTAX. Como a tabela ai_billing possui apenas o mês da cobrança e não a data exata da fatura, foi utilizada a média das cotações diárias de venda de cada mês. Essa escolha é uma aproximação explícita e auditável. Caso a data exata da cobrança esteja disponível futuramente, a conversão poderá ser realizada pela cotação correspondente àquela data.

Portal do BCB:  
https://dadosabertos.bcb.gov.br/dataset/1-taxa-de-cambio---livre---dolar-americano-venda---diario

Como `ai_billing` possui apenas o mês da cobrança, e não o dia exato da fatura, usamos a **média das cotações diárias de venda em cada mês**. Essa é uma aproximação explícita. Se futuramente houver a data exata da cobrança, a conversão deverá usar a PTAX correspondente àquela data.


In [10]:
import requests
import pandas as pd
import time

billing = clean["ai_billing.csv"].copy()

# Intervalo necessário de acordo com a base
mes_min = pd.Period(billing["billing_month"].min(), freq="M")
mes_max = pd.Period(billing["billing_month"].max(), freq="M")

# A API PTAX usa MM-DD-AAAA
data_inicial = mes_min.start_time.strftime("%m-%d-%Y")
data_final = mes_max.end_time.strftime("%m-%d-%Y")

url_ptax = (
    "https://olinda.bcb.gov.br/olinda/servico/PTAX/versao/v1/"
    "odata/CotacaoDolarPeriodo"
    f"(dataInicial=@dataInicial,dataFinalCotacao=@dataFinalCotacao)"
    f"?@dataInicial='{data_inicial}'"
    f"&@dataFinalCotacao='{data_final}'"
    "&$format=json"
)

# Faz algumas tentativas caso a API demore
resposta = None

for tentativa in range(3):
    try:
        resposta = requests.get(url_ptax, timeout=90)
        resposta.raise_for_status()
        break

    except requests.exceptions.RequestException as erro:
        print(f"Tentativa {tentativa + 1} falhou: {erro}")

        if tentativa < 2:
            print("Tentando novamente...")
            time.sleep(5)
        else:
            raise

dados = resposta.json()["value"]

ptax_diaria = pd.DataFrame(dados)

if ptax_diaria.empty:
    raise ValueError(
        "A API PTAX do Banco Central não retornou cotações para o período."
    )

display(ptax_diaria.head())

# Converte a data/hora retornada pela PTAX
ptax_diaria["dataHoraCotacao"] = pd.to_datetime(
    ptax_diaria["dataHoraCotacao"],
    errors="coerce"
)

# Cria o mês de referência
ptax_diaria["billing_month"] = (
    ptax_diaria["dataHoraCotacao"]
    .dt.to_period("M")
    .astype(str)
)

# Como pode haver mais de uma cotação no mesmo dia,
# primeiro calculamos a média diária da cotação de venda
ptax_por_dia = (
    ptax_diaria
    .assign(data=ptax_diaria["dataHoraCotacao"].dt.date)
    .groupby(["data", "billing_month"], as_index=False)["cotacaoVenda"]
    .mean()
)

# Depois calculamos a média mensal
ptax_mensal = (
    ptax_por_dia
    .groupby("billing_month", as_index=False)["cotacaoVenda"]
    .mean()
    .rename(columns={"cotacaoVenda": "fx_rate_to_brl"})
)

display(ptax_mensal)

,cotacaoCompra,cotacaoVenda,dataHoraCotacao
0,5.6931,5.6937,2025-06-02 13:09:05.107
1,5.6688,5.6694,2025-06-03 13:07:29.518
2,5.6326,5.6332,2025-06-04 13:11:29.327
3,5.5963,5.5969,2025-06-05 13:07:30.991
4,5.5941,5.5947,2025-06-06 13:08:48.677


,billing_month,fx_rate_to_brl
0,2025-06,5.547055
1,2025-07,5.528491
2,2025-08,5.446924
3,2025-09,5.367409
4,2025-10,5.385526
5,2025-11,5.340853
6,2025-12,5.453091
7,2026-01,5.338024
8,2026-02,5.200617
9,2026-03,5.231586


In [11]:
billing["currency_original"] = billing["currency"]
billing["total_cost_original"] = billing["total_cost"]

billing = billing.merge(
    ptax_mensal,
    on="billing_month",
    how="left"
)

# Para valores que já estão em BRL, a taxa é 1.
billing.loc[billing["currency"].eq("BRL"), "fx_rate_to_brl"] = 1.0

# Verificação: nenhum lançamento em USD pode ficar sem cotação.
sem_cotacao = billing.loc[
    billing["currency"].eq("USD") & billing["fx_rate_to_brl"].isna()
]

if len(sem_cotacao) > 0:
    raise ValueError(
        f"Há {len(sem_cotacao)} lançamentos em USD sem taxa de câmbio."
    )

billing["total_cost_brl"] = (
    billing["total_cost_original"] * billing["fx_rate_to_brl"]
).round(2)

billing["currency_standard"] = "BRL"

clean["ai_billing.csv"] = billing

display(
    billing[
        [
            "billing_month",
            "total_cost_original",
            "currency_original",
            "fx_rate_to_brl",
            "total_cost_brl",
            "currency_standard"
        ]
    ].head(10)
)


,billing_month,total_cost_original,currency_original,fx_rate_to_brl,total_cost_brl,currency_standard
0,2025-06,1.87,USD,5.547055,10.37,BRL
1,2025-06,0.74,USD,5.547055,4.10,BRL
2,2025-06,2.73,USD,5.547055,15.14,BRL
3,2025-06,11.46,USD,5.547055,63.57,BRL
4,2025-06,0.33,USD,5.547055,1.83,BRL
5,2025-06,1.22,USD,5.547055,6.77,BRL
6,2025-06,0.78,USD,5.547055,4.33,BRL
7,2025-06,0.63,USD,5.547055,3.49,BRL
8,2025-06,0.11,USD,5.547055,0.61,BRL
9,2025-06,1.22,USD,5.547055,6.77,BRL


### Por que preservar os campos originais?

`total_cost_original` e `currency_original` permanecem intactos. A taxa aplicada também é armazenada em `fx_rate_to_brl`.

Isso torna a conversão:

- auditável;
- reversível;
- reproduzível;
- atualizável sem perda do dado bruto.


## 5. Formulação do problema preditivo

O feedback do Deliverable 1 pede quatro respostas explícitas.

### 5.1 O que será previsto?

Será previsto o **Risk Score de Dependência de IA do colaborador no mês seguinte**, em escala contínua de **0 a 100**.

O score representa uma **proxy operacional de risco**, porque a base não contém um rótulo observado de dependência.

### 5.2 Qual é a unidade de análise?

**Colaborador × mês.**

Cada linha da base analítica representa o comportamento de um colaborador durante um mês.

### 5.3 Qual é a tarefa?

**Regressão.**

O alvo é contínuo, variando de 0 a 100.

### 5.4 Qual é a métrica principal?

**MAE — Mean Absolute Error (Erro Absoluto Médio).**

O MAE é a métrica principal porque está na mesma unidade do score. Assim, se o MAE for 6, por exemplo, significa que a previsão erra em média cerca de 6 pontos em uma escala de 0 a 100.

Isso corresponde diretamente à decisão do gestor: o score será usado para **priorizar colaboradores com maior risco para análise, acompanhamento e ações de governança**. Uma métrica expressa em pontos do próprio score é simples de interpretar nesse contexto.

`R²` pode ser apresentado como métrica complementar, mas não substitui o MAE como métrica principal.


## 6. Construção da unidade colaborador × mês

In [12]:
usage = clean["ai_usage_logs.csv"].copy()

usage["year_month"] = usage["timestamp"].dt.to_period("M").astype(str)
usage["hour"] = usage["timestamp"].dt.hour
usage["day_of_week"] = usage["timestamp"].dt.dayofweek

usage["latency_missing"] = usage["latency_ms"].isna().astype(int)
usage["is_failure"] = (usage["request_status"] != "success").astype(int)
usage["total_tokens"] = usage["input_tokens"] + usage["output_tokens"]

# Indicadores temporais observáveis no próprio mês.
usage["is_weekend"] = (usage["day_of_week"] >= 5).astype(int)
usage["is_off_hours"] = (
    (usage["hour"] < 8) | (usage["hour"] >= 19)
).astype(int)

usage.head()


,usage_id,timestamp,employee_id,company_id,department_id,ai_tool_id,model_id,application,request_count,input_tokens,...,project_id,cost_center,year_month,hour,day_of_week,latency_missing,is_failure,total_tokens,is_weekend,is_off_hours
0,U0000001,2026-03-27 21:26:54,C01-E0001,C01,C01-D02,T07,M10,Extensao VSCode,18,1526,...,C01-P01,CC-C01-D02,2026-03,21,4,0,0,4959,0,1
1,U0000002,2026-04-08 11:09:07,C01-E0001,C01,C01-D02,T07,M10,Extensao VSCode,32,3610,...,C01-P03,CC-C01-D02,2026-04,11,2,0,0,6210,0,0
2,U0000003,2026-04-07 12:12:04,C01-E0001,C01,C01-D02,T07,M10,Extensao VSCode,18,1226,...,C01-P01,CC-C01-D02,2026-04,12,1,0,0,4740,0,0
3,U0000004,2026-04-07 21:34:24,C01-E0001,C01,C01-D02,T02,M04,Chat Web,1,235,...,NaN,CC-C01-D02,2026-04,21,1,0,0,982,0,1
4,U0000005,2026-05-26 14:07:13,C01-E0001,C01,C01-D02,T07,M10,Extensao VSCode,33,3821,...,C01-P03,CC-C01-D02,2026-05,14,1,0,0,5231,0,0


In [13]:
# Concentração por ferramenta (HHI) dentro de cada colaborador/mês.
tool_count = (
    usage
    .groupby(["employee_id", "year_month", "ai_tool_id"])
    .size()
    .rename("n")
    .reset_index()
)

tool_count["total_mes"] = (
    tool_count
    .groupby(["employee_id", "year_month"])["n"]
    .transform("sum")
)

tool_count["share2"] = (
    tool_count["n"] / tool_count["total_mes"]
) ** 2

tool_hhi = (
    tool_count
    .groupby(["employee_id", "year_month"], as_index=False)["share2"]
    .sum()
    .rename(columns={"share2": "tool_hhi"})
)

mensal = (
    usage
    .groupby(["employee_id", "year_month"], as_index=False)
    .agg(
        usage_events=("usage_id", "count"),
        requests_total=("request_count", "sum"),
        total_tokens=("total_tokens", "sum"),
        active_days=("timestamp", lambda s: s.dt.date.nunique()),
        distinct_tools=("ai_tool_id", "nunique"),
        distinct_applications=("application", "nunique"),
        latency_ms_mean=("latency_ms", "mean"),
        latency_missing_rate=("latency_missing", "mean"),
        failure_rate=("is_failure", "mean"),
        weekend_usage_rate=("is_weekend", "mean"),
        off_hours_usage_rate=("is_off_hours", "mean"),
    )
    .merge(tool_hhi, on=["employee_id", "year_month"], how="left")
)

mensal["requests_per_active_day"] = (
    mensal["requests_total"] /
    mensal["active_days"].replace(0, np.nan)
).fillna(0)

display(mensal.head())


,employee_id,year_month,usage_events,requests_total,total_tokens,active_days,distinct_tools,distinct_applications,latency_ms_mean,latency_missing_rate,failure_rate,weekend_usage_rate,off_hours_usage_rate,tool_hhi,requests_per_active_day
0,C01-E0001,2026-03,1,18,4959,1,1,1,261.000000,0.0,0.000000,0.00,1.000000,1.000000,18.000000
1,C01-E0001,2026-04,3,51,11932,2,2,2,958.333333,0.0,0.000000,0.00,0.333333,0.555556,25.500000
2,C01-E0001,2026-05,1,33,5231,1,1,1,75.000000,0.0,0.000000,0.00,0.000000,1.000000,33.000000
3,C01-E0001,2026-06,8,76,30709,8,5,5,880.750000,0.0,0.000000,0.25,0.250000,0.250000,9.500000
4,C01-E0001,2026-07,3,47,21332,3,3,3,788.666667,0.0,0.333333,0.00,0.333333,0.333333,15.666667


### Por que essa granularidade?

O produto pretende mostrar risco para gestores. Um log isolado representa apenas uma requisição; ele não representa o padrão de comportamento de uma pessoa.

A unidade **colaborador × mês** permite observar intensidade, frequência e concentração do uso ao longo de um período suficientemente estável para gerar um indicador gerencial.


## 7. Definição operacional do Risk Score 0–100

Como não existe rótulo de dependência na base, o score não pode ser apresentado como “verdade observada”. Ele é uma **proxy operacional proposta pelo grupo**.

Para evitar pesos arbitrários diferentes entre variáveis, usamos quatro componentes com **peso igual (25% cada)**:

1. **Frequência de uso:** `active_days`;
2. **Intensidade:** `requests_per_active_day`;
3. **Volume computacional:** `total_tokens`;
4. **Concentração em ferramentas:** `tool_hhi`.

Cada componente é transformado em **percentil dentro do próprio mês** (0 a 100). O score é a média simples dos quatro percentis.

A escolha é propositalmente simples, transparente e auditável. Ela não utiliza variáveis que não existem na base, como revisão humana ou repetição textual de prompts.


In [14]:
componentes_score = [
    "active_days",
    "requests_per_active_day",
    "total_tokens",
    "tool_hhi"
]

for coluna in componentes_score:
    mensal[f"{coluna}_pct"] = (
        mensal
        .groupby("year_month")[coluna]
        .rank(method="average", pct=True)
        * 100
    )

mensal["risk_score"] = (
    mensal[[f"{c}_pct" for c in componentes_score]]
    .mean(axis=1)
    .clip(0, 100)
)

display(
    mensal[
        ["employee_id", "year_month"] +
        componentes_score +
        ["risk_score"]
    ].head(10)
)

print(mensal["risk_score"].describe())


,employee_id,year_month,active_days,requests_per_active_day,total_tokens,tool_hhi,risk_score
0,C01-E0001,2026-03,1,18.000000,4959,1.000000,36.417323
1,C01-E0001,2026-04,2,25.500000,11932,0.555556,37.560916
2,C01-E0001,2026-05,1,33.000000,5231,1.000000,40.367861
3,C01-E0001,2026-06,8,9.500000,30709,0.250000,20.309501
4,C01-E0001,2026-07,3,15.666667,21332,0.333333,25.096339
5,C01-E0003,2025-06,3,15.666667,55065,0.428571,33.878968
6,C01-E0003,2025-07,3,4.333333,24813,1.000000,33.441311
7,C01-E0003,2025-08,6,12.166667,107805,0.306122,33.956693
8,C01-E0003,2025-09,5,15.200000,185840,0.280000,32.818533
9,C01-E0003,2025-10,2,41.500000,186660,0.500000,48.611111


count    12438.000000
mean        50.056279
std         16.138193
min          9.697421
25%         36.325530
50%         49.567863
75%         64.234744
max         87.550201
Name: risk_score, dtype: float64


### Limitação metodológica

O score foi construído com comportamento de uso porque são os sinais observáveis disponíveis. Portanto, ele mede **risco operacional de dependência segundo a definição do projeto**, não dependência psicológica, clínica ou causal.

A definição deve ser validada com o professor e com stakeholders antes de ser usada como indicador definitivo do produto.


## 8. Alvo: risco no mês seguinte

In [15]:
mensal = mensal.sort_values(["employee_id", "year_month"]).copy()

# Converte o mês para Period para garantir que o próximo registro seja realmente o mês seguinte.
mensal["period"] = pd.PeriodIndex(mensal["year_month"], freq="M")

mensal["next_period"] = (
    mensal
    .groupby("employee_id")["period"]
    .shift(-1)
)

mensal["risk_score_next_month"] = (
    mensal
    .groupby("employee_id")["risk_score"]
    .shift(-1)
)

# Mantém somente pares de meses consecutivos.
base_modelagem = mensal[
    mensal["next_period"].eq(mensal["period"] + 1)
].copy()

# O alvo é explicitamente o score do mês seguinte.
ALVO = "risk_score_next_month"

print("Unidade de análise: colaborador x mês")
print("Tarefa: regressão")
print("Alvo:", ALVO)
print("Linhas disponíveis para modelagem:", len(base_modelagem))

display(
    base_modelagem[
        ["employee_id", "year_month", "risk_score", ALVO]
    ].head()
)


Unidade de análise: colaborador x mês
Tarefa: regressão
Alvo: risk_score_next_month
Linhas disponíveis para modelagem: 11168


,employee_id,year_month,risk_score,risk_score_next_month
0,C01-E0001,2026-03,36.417323,37.560916
1,C01-E0001,2026-04,37.560916,40.367861
2,C01-E0001,2026-05,40.367861,20.309501
3,C01-E0001,2026-06,20.309501,25.096339
5,C01-E0003,2025-06,33.878968,33.441311


### Por que deslocar o alvo?

Para prever o risco do próximo mês, os preditores precisam representar somente o que já é conhecido no mês atual.

Exemplo:

- atributos de março → previsão do score de abril;
- atributos de abril → previsão do score de maio.

Isso separa claramente **momento da previsão** e **resultado futuro**.


## 9. Preditores disponíveis no momento da previsão

In [16]:
FEATURES = [
    "usage_events",
    "requests_total",
    "total_tokens",
    "active_days",
    "requests_per_active_day",
    "distinct_tools",
    "distinct_applications",
    "latency_ms_mean",
    "latency_missing_rate",
    "failure_rate",
    "weekend_usage_rate",
    "off_hours_usage_rate",
    "tool_hhi"
]

X = base_modelagem[FEATURES].copy()
y = base_modelagem[ALVO].copy()

print("Preditores usados:")
for c in FEATURES:
    print("-", c)


Preditores usados:
- usage_events
- requests_total
- total_tokens
- active_days
- requests_per_active_day
- distinct_tools
- distinct_applications
- latency_ms_mean
- latency_missing_rate
- failure_rate
- weekend_usage_rate
- off_hours_usage_rate
- tool_hhi


### Verificação de disponibilidade

Todos os preditores acima são calculados somente com eventos ocorridos **até o fim do mês atual**. Nenhum deles utiliza o `risk_score` do mês seguinte.

Também evitamos, nesta versão inicial, variáveis cadastrais que podem representar o estado atual do sistema, mas não possuem histórico versionado — por exemplo, `active`, cargo atual ou senioridade atual. Usá-las retrospectivamente poderia introduzir informação que talvez não estivesse disponível naquele momento passado.

Da mesma forma, `employee_id` é mantido apenas como chave de rastreabilidade e **não entra como preditor**.


## 10. Corte temporal

A divisão não será aleatória. O objetivo é simular o uso real: treinar com o passado e avaliar em meses posteriores.

Considerando os meses disponíveis na base e lembrando que cada linha usa o mês atual para prever o seguinte:

- **Treino:** meses de referência de 2025-06 até 2026-02;
- **Validação:** meses de referência 2026-03 e 2026-04;
- **Teste final:** meses de referência 2026-05 e 2026-06.

Assim, o teste final utiliza informações de maio e junho para prever, respectivamente, junho e julho de 2026.


In [17]:
periodo_ref = pd.PeriodIndex(base_modelagem["year_month"], freq="M")

mask_treino = periodo_ref <= pd.Period("2026-02", freq="M")
mask_valid = periodo_ref.isin([
    pd.Period("2026-03", freq="M"),
    pd.Period("2026-04", freq="M")
])
mask_teste = periodo_ref.isin([
    pd.Period("2026-05", freq="M"),
    pd.Period("2026-06", freq="M")
])

treino = base_modelagem.loc[mask_treino].copy()
validacao = base_modelagem.loc[mask_valid].copy()
teste = base_modelagem.loc[mask_teste].copy()

print("Treino:", treino["year_month"].min(), "a", treino["year_month"].max(), "-", len(treino), "linhas")
print("Validação:", validacao["year_month"].min(), "a", validacao["year_month"].max(), "-", len(validacao), "linhas")
print("Teste:", teste["year_month"].min(), "a", teste["year_month"].max(), "-", len(teste), "linhas")


Treino: 2025-06 a 2026-02 - 7112 linhas
Validação: 2026-03 a 2026-04 - 2009 linhas
Teste: 2026-05 a 2026-06 - 2047 linhas


### Por que o corte é temporal?

Em produção, o gestor sempre tentará prever um período que ainda não aconteceu. Uma divisão aleatória permitiria que meses futuros participassem do treinamento usado para avaliar meses passados, produzindo uma estimativa otimista da performance.


## 11. Pipeline de preparação sem vazamento

In [18]:
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import RobustScaler

# Todas as variáveis desta primeira versão são numéricas.
# Imputação e escala são aprendidas APENAS no treino.
numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median", add_indicator=True)),
    ("scaler", RobustScaler())
])

preprocessor = ColumnTransformer([
    ("num", numeric_pipeline, FEATURES)
])

preprocessor


ColumnTransformer(transformers=[('num',
                                 Pipeline(steps=[('imputer',
                                                  SimpleImputer(add_indicator=True,
                                                                strategy='median')),
                                                 ('scaler', RobustScaler())]),
                                 ['usage_events', 'requests_total',
                                  'total_tokens', 'active_days',
                                  'requests_per_active_day', 'distinct_tools',
                                  'distinct_applications', 'latency_ms_mean',
                                  'latency_missing_rate', 'failure_rate',
                                  'weekend_usage_rate', 'off_hours_usage_rate',
                                  'tool_hhi'])])

O `fit` do pré-processamento deve ser feito somente sobre `treino`. Validação e teste recebem apenas `transform`.

Isso impede que mediana, escala ou qualquer outro parâmetro de preparação sejam aprendidos com meses futuros.


## 12. Baseline simples para tornar a métrica concreta

In [19]:
from sklearn.dummy import DummyRegressor
from sklearn.metrics import mean_absolute_error, r2_score

X_train = treino[FEATURES]
y_train = treino[ALVO]

X_valid = validacao[FEATURES]
y_valid = validacao[ALVO]

baseline = Pipeline([
    ("prep", preprocessor),
    ("model", DummyRegressor(strategy="median"))
])

baseline.fit(X_train, y_train)

pred_valid = baseline.predict(X_valid)

mae_baseline = mean_absolute_error(y_valid, pred_valid)
r2_baseline = r2_score(y_valid, pred_valid)

print(f"MAE baseline (validação): {mae_baseline:.2f} pontos")
print(f"R² baseline (validação): {r2_baseline:.4f}")


MAE baseline (validação): 14.19 pontos
R² baseline (validação): -0.0011


### Interpretação

O `DummyRegressor` não é o modelo final. Ele serve apenas como **linha de base**.

Qualquer algoritmo escolhido posteriormente deverá ser comparado a esse resultado. O objetivo é demonstrar ganho real em relação a simplesmente prever um valor central do histórico.

A métrica principal continuará sendo o **MAE**.


## 13. Registro das principais decisões

In [20]:
decisoes = pd.DataFrame([
    {
        "item": "Unidade de análise",
        "decisao": "Colaborador x mês",
        "justificativa": "Representa comportamento agregado e é compatível com a decisão gerencial."
    },
    {
        "item": "Variável-alvo",
        "decisao": "Risk Score 0–100 do mês seguinte",
        "justificativa": "Permite antecipar risco futuro em vez de apenas descrever o mês corrente."
    },
    {
        "item": "Tarefa",
        "decisao": "Regressão",
        "justificativa": "O alvo é contínuo."
    },
    {
        "item": "Métrica principal",
        "decisao": "MAE",
        "justificativa": "É expresso diretamente em pontos do Risk Score e é fácil de interpretar pelo gestor."
    },
    {
        "item": "Corte",
        "decisao": "Temporal",
        "justificativa": "Simula previsão futura e evita contaminação entre passado e futuro."
    },
    {
        "item": "latency_ms",
        "decisao": "Preservar NaN + indicador de ausência",
        "justificativa": "A ausência se concentra em falhas e pode ser sinal."
    },
    {
        "item": "NaNs estruturais",
        "decisao": "Não imputar automaticamente",
        "justificativa": "Em diversos campos, NaN significa que a informação não se aplica."
    },
    {
        "item": "Câmbio",
        "decisao": "Média mensal da série diária oficial do BCB (SGS 1)",
        "justificativa": "A base possui somente mês da cobrança e a fonte oficial torna a conversão rastreável."
    },
    {
        "item": "Valores financeiros originais",
        "decisao": "Preservar",
        "justificativa": "Mantém auditoria, reversibilidade e reprodutibilidade."
    },
    {
        "item": "IDs",
        "decisao": "Rastreabilidade, não predição",
        "justificativa": "Evita memorização de indivíduos e explosão de cardinalidade."
    }
])

display(decisoes)


,item,decisao,justificativa
0,Unidade de análise,Colaborador x mês,Representa comportamento agregado e é compatív...
1,Variável-alvo,Risk Score 0–100 do mês seguinte,Permite antecipar risco futuro em vez de apena...
2,Tarefa,Regressão,O alvo é contínuo.
3,Métrica principal,MAE,É expresso diretamente em pontos do Risk Score...
4,Corte,Temporal,Simula previsão futura e evita contaminação en...
5,latency_ms,Preservar NaN + indicador de ausência,A ausência se concentra em falhas e pode ser s...
6,NaNs estruturais,Não imputar automaticamente,"Em diversos campos, NaN significa que a inform..."
7,Câmbio,Média mensal da série diária oficial do BCB (S...,A base possui somente mês da cobrança e a font...
8,Valores financeiros originais,Preservar,"Mantém auditoria, reversibilidade e reprodutib..."
9,IDs,"Rastreabilidade, não predição",Evita memorização de indivíduos e explosão de ...


## 14. Conclusão

Após os ajustes do feedback, o problema analítico passa a estar formulado de maneira explícita:

> **Usar o comportamento de uso de IA observado para cada colaborador no mês atual para prever seu Risk Score operacional de dependência de IA no mês seguinte.**

- **Unidade:** colaborador × mês;
- **Alvo:** `risk_score_next_month`, contínuo de 0 a 100;
- **Tarefa:** regressão;
- **Métrica principal:** MAE;
- **Decisão apoiada:** priorização de colaboradores com maior risco previsto para acompanhamento e ações de governança;
- **Validação:** temporal;
- **Preditores:** somente informações disponíveis até o fim do mês de referência;
- **Câmbio**: série oficial do Banco Central do Brasil, com origem documentada.
